In [1]:
# ============================================================
# Cell 1 - Install Required Libraries
# ============================================================

# ---------- LangChain ----------
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-core
%pip install -U langchain-text-splitters
%pip install -U langchain-huggingface
%pip install -U langchain-ollama

# ---------- PDF Loader ----------
%pip install -U unstructured
%pip install -U pypdf

# ---------- Embedding Model ----------
%pip install -U sentence-transformers

# ---------- Communicate with Solr ----------
%pip install -U requests
%pip install -U sentence-transformers
print("✅ All libraries installed successfully.")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ All libraries installed successfully.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# Cell 2 - Imports and Configuration
# ============================================================

import os
import glob
import requests

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama


# PDF folder
PDF_FOLDER = "documents"


# Solr configuration
SOLR_HOST = "http://localhost:8983"
SOLR_CORE = "rag_documents"
SOLR_URL = f"{SOLR_HOST}/solr/{SOLR_CORE}"


# Chunking configuration
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50


# Model configuration
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

OLLAMA_MODEL_NAME = "qwen2.5:3b"


print("✅ Imports and configuration completed")
print("Solr URL:", SOLR_URL)
print("PDF folder:", PDF_FOLDER)

C:\Users\AbidDileepJan\AppData\Local\Temp\ipykernel_24092\801613444.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\AbidDileepJan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports and configuration completed
Solr URL: http://localhost:8983/solr/rag_documents
PDF folder: documents


In [3]:
# ============================================================
# Cell 3 - Basic Setup Check
# ============================================================

import os
import requests


# ------------------------------------------------------------
# Check the PDF folder
# ------------------------------------------------------------

assert os.path.isdir(PDF_FOLDER), (
    f"PDF folder not found: {PDF_FOLDER}"
)

pdf_files = [
    file_name
    for file_name in os.listdir(PDF_FOLDER)
    if file_name.lower().endswith(".pdf")
]

assert pdf_files, (
    f"No PDF files found inside: {PDF_FOLDER}"
)

print("✅ PDF folder found:", PDF_FOLDER)
print("✅ Number of PDF files:", len(pdf_files))


# ------------------------------------------------------------
# Check the Solr core
# ------------------------------------------------------------

response = requests.get(
    f"{SOLR_URL}/admin/ping",
    timeout=10
)

response.raise_for_status()

print("✅ Solr core is running:", SOLR_CORE)

✅ PDF folder found: documents
✅ Number of PDF files: 2
✅ Solr core is running: rag_documents


In [4]:
# ============================================================
# Load All PDFs from documents Folder
# ============================================================

import os
import glob
from langchain_community.document_loaders import PyPDFLoader

PDF_FOLDER = "documents"

pdf_files = glob.glob(os.path.join(PDF_FOLDER, "*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in '{PDF_FOLDER}'"
    )

print(f"Found {len(pdf_files)} PDF file(s):\n")

documents = []

for pdf in pdf_files:

    print(f"Loading: {os.path.basename(pdf)}")

    loader = PyPDFLoader(pdf)

    pdf_documents = loader.load()

    print(f"   Pages: {len(pdf_documents)}")

    documents.extend(pdf_documents)

print("\n" + "=" * 50)
print(f"Total PDFs Loaded : {len(pdf_files)}")
print(f"Total Pages Loaded: {len(documents)}")
print("=" * 50)

Found 2 PDF file(s):

Loading: Oil and gas production handbook ed3x0_web.pdf
   Pages: 162
Loading: US10655422.pdf
   Pages: 18

Total PDFs Loaded : 2
Total Pages Loaded: 180


In [5]:
# ============================================================
# Cell 4 - Create Solr Schema
# ============================================================

SCHEMA_URL = f"{SOLR_URL}/schema"

schema_commands = {
    "add-field-type": {
        "name": "knn_vector_384",
        "class": "solr.DenseVectorField",
        "vectorDimension": 384,
        "similarityFunction": "cosine"
    },
    "add-field": [
        {
            "name": "content",
            "type": "text_general",
            "indexed": True,
            "stored": True
        },
        {
            "name": "source",
            "type": "string",
            "indexed": True,
            "stored": True
        },
        {
            "name": "page",
            "type": "pint",
            "indexed": True,
            "stored": True
        },
        {
            "name": "vector",
            "type": "knn_vector_384",
            "indexed": True,
            "stored": True
        }
    ]
}

response = requests.post(
    SCHEMA_URL,
    json=schema_commands,
    timeout=30
)

if response.status_code == 200:
    print("✅ Solr schema created successfully.")
else:
    print("❌ Schema creation failed.")
    print("Status code:", response.status_code)
    print("Response:", response.text)

❌ Schema creation failed.
Status code: 400
Response: {
  "responseHeader":{
    "status":400,
    "QTime":6
  },
  "error":{
    "metadata":["error-class","org.apache.solr.api.ApiBag$ExceptionWithErrObject","root-error-class","org.apache.solr.api.ApiBag$ExceptionWithErrObject"],
    "details":[{
      "add-field-type":{
        "name":"knn_vector_384",
        "class":"solr.DenseVectorField",
        "vectorDimension":384,
        "similarityFunction":"cosine"
      },
      "errorMessages":["Field type 'knn_vector_384' already exists.\n"]
    },{
      "errorMessages":["Field 'content' already exists.\n"],
      "add-field":{
        "name":"content",
        "type":"text_general",
        "indexed":true,
        "stored":true
      }
    },{
      "errorMessages":["Field 'source' already exists.\n"],
      "add-field":{
        "name":"source",
        "type":"string",
        "indexed":true,
        "stored":true
      }
    },{
      "errorMessages":["Field 'page' already exists.\n

In [6]:
# ============================================================
# Delete Old Documents from Solr
# ============================================================

response = requests.post(
    f"{SOLR_URL}/update",
    params={"commit": "true"},
    json={
        "delete": {
            "query": "*:*"
        }
    }
)

if response.status_code == 200:
    print("✅ Old documents deleted successfully")
else:
    print("❌ Delete failed")
    print("Status code:", response.status_code)
    print("Response:", response.text)

✅ Old documents deleted successfully


In [7]:
# ============================================================
# Cell 5 - Chunk All Loaded PDF Documents
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter


# Make sure documents were loaded
if not documents:
    raise ValueError(
        "No PDF pages are available. Run the PDF loading cell first."
    )


# Create the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)


# Split all loaded PDF pages into chunks
chunks = text_splitter.split_documents(
    documents
)


print(f"Total pages loaded   : {len(documents)}")
print(f"Total chunks created : {len(chunks)}")


# Show one sample chunk
if chunks:

    print("\n" + "=" * 70)
    print("Sample Chunk")
    print("=" * 70)

    print(chunks[0].page_content[:500])

    print("\nMetadata:")
    print(chunks[0].metadata)

Total pages loaded   : 180
Total chunks created : 962

Sample Chunk
Oil and gas production handbook  
An introduction to oil and gas production, 
transport, refining and petrochemical 
industry
Håvard Devold

Metadata:
{'producer': 'Adobe Acrobat 8.3', 'creator': 'Adobe InDesign CS5.5 (7.5.3)', 'creationdate': '2013-08-22T11:22:31+02:00', 'moddate': '2013-08-22T11:22:31+02:00', 'source': 'documents\\Oil and gas production handbook ed3x0_web.pdf', 'total_pages': 162, 'page': 0, 'page_label': '1'}


In [8]:
# ============================================================
# Cell 6 - Generate Embeddings
# ============================================================

# Load the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

# Extract text from every chunk
chunk_texts = [
    chunk.page_content
    for chunk in chunks
]

# Convert all chunk texts into embedding vectors
chunk_vectors = embedding_model.embed_documents(chunk_texts)

print("✅ Embeddings generated successfully")
print("Total embeddings:", len(chunk_vectors))
print("Dimension of each embedding:", len(chunk_vectors[0]))

# Display a small part of the first vector
print("\nFirst 10 values of the first embedding:")
print(chunk_vectors[0][:10])



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6199.52it/s]


✅ Embeddings generated successfully
Total embeddings: 962
Dimension of each embedding: 384

First 10 values of the first embedding:
[-0.056089553982019424, -0.03441007062792778, -0.04012473300099373, -0.016032172366976738, -0.043585553765296936, 0.011881006881594658, -0.054741378873586655, 0.05803079903125763, -0.09454365819692612, -0.09625495970249176]


In [9]:
# ============================================================
# Upload Chunks and Embeddings to Solr
# ============================================================

solr_documents = []

for index, (chunk, vector) in enumerate(
    zip(chunks, chunk_vectors)
):

    source_path = chunk.metadata.get(
        "source",
        "Unknown"
    )

    page_number = chunk.metadata.get(
        "page",
        0
    )

    solr_document = {
        "id": f"chunk_{index}",
        "content": chunk.page_content,
        "source": os.path.basename(source_path),
        "page": int(page_number) + 1,
        "vector": [
            float(value)
            for value in vector
        ]
    }

    solr_documents.append(solr_document)


if not solr_documents:
    raise ValueError(
        "No documents were prepared for Solr upload."
    )


print("Documents prepared:", len(solr_documents))
print(
    "Vector dimension:",
    len(solr_documents[0]["vector"])
)


response = requests.post(
    f"{SOLR_URL}/update",
    params={"commit": "true"},
    json=solr_documents,
    timeout=120
)

if response.status_code == 200:
    print("✅ Upload completed successfully")
else:
    print("❌ Upload failed")
    print("Status code:", response.status_code)
    print("Response:", response.text)

response.raise_for_status()

Documents prepared: 962
Vector dimension: 384
✅ Upload completed successfully


In [10]:
# ============================================================
# Cell 7 - Load Cross Encoder Reranker
# ============================================================

from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = (
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    device="cpu"
)

print("✅ Reranker loaded successfully")
print("Model:", RERANKER_MODEL_NAME)

c:\Users\AbidDileepJan\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AbidDileepJan\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10449.

✅ Reranker loaded successfully
Model: cross-encoder/ms-marco-MiniLM-L6-v2


In [11]:
# ============================================================
# Cell 8 - Reranker Configuration
# ============================================================

RETRIEVAL_TOP_K = 6
RERANK_TOP_K = 2

print("Solr candidate chunks:", RETRIEVAL_TOP_K)
print("Final reranked chunks:", RERANK_TOP_K)

Solr candidate chunks: 6
Final reranked chunks: 2


In [12]:
# ============================================================
# Hybrid Search Configuration
# ============================================================

VECTOR_TOP_K = 6
KEYWORD_TOP_K = 6
HYBRID_TOP_K = 10

print("Hybrid search configuration created")

Hybrid search configuration created


In [13]:
# ============================================================
# Cell 8 - Retrieve and Rerank Relevant Chunks from Solr
# ============================================================

def retrieve_chunks(
    question,
    top_k=RETRIEVAL_TOP_K
):
    """
    Retrieve relevant chunks from Solr using vector search.
    """

    # Generate embedding for the question
    question_vector = embedding_model.embed_query(
        question
    )

    # Convert NumPy values to normal Python floats
    question_vector = [
        float(value)
        for value in question_vector
    ]

    # Send the vector in the POST request body
    response = requests.post(
        f"{SOLR_URL}/select",
        json={
            "query": (
                f"{{!knn f=vector topK={top_k}}}"
                f"{question_vector}"
            ),
            "fields": "content,source,page,score",
            "limit": top_k
        },
        timeout=60
    )

    if response.status_code != 200:
        print("❌ Vector search failed")
        print("Status code:", response.status_code)
        print("Response:", response.text)

    response.raise_for_status()

    results = response.json()["response"]["docs"]

    for result in results:
        result["search_type"] = "vector"

    return results


def rerank_chunks(
    question,
    retrieved_chunks,
    top_k=RERANK_TOP_K
):
    """
    Rerank retrieved chunks using the Cross-Encoder model.
    """

    if not retrieved_chunks:
        return []

    # Create question and chunk pairs
    question_chunk_pairs = [
        (
            question,
            chunk.get("content", "")
        )
        for chunk in retrieved_chunks
    ]

    # Generate Cross-Encoder scores
    reranker_scores = reranker.predict(
        question_chunk_pairs
    )

    reranked_results = []

    for chunk, score in zip(
        retrieved_chunks,
        reranker_scores
    ):
        result = chunk.copy()
        result["reranker_score"] = float(score)

        reranked_results.append(result)

    # Highest reranker score first
    reranked_results.sort(
        key=lambda item: item["reranker_score"],
        reverse=True
    )

    return reranked_results[:top_k]


print("✅ Vector retrieval and reranking functions created")

✅ Vector retrieval and reranking functions created


In [14]:
# ============================================================
# Keyword Search Function
# ============================================================

def keyword_search(
    question,
    top_k=KEYWORD_TOP_K
):
    """
    Retrieve chunks from Solr using BM25 keyword search.
    """

    response = requests.get(
        f"{SOLR_URL}/select",
        params={
            "q": question,
            "qf": "content",
            "fl": "content,source,page,score",
            "rows": top_k,
            "defType": "edismax"
        },
        timeout=60
    )

    response.raise_for_status()

    results = response.json()["response"]["docs"]

    # Mark results as keyword-search results
    for result in results:
        result["search_type"] = "keyword"

    return results


print("✅ Keyword search function created")

✅ Keyword search function created


In [23]:
# ============================================================
# Hybrid Search Function
# Vector Search + Keyword Search
# ============================================================

def hybrid_search(
    question,
    vector_top_k=VECTOR_TOP_K,
    keyword_top_k=KEYWORD_TOP_K,
    hybrid_top_k=HYBRID_TOP_K
):
    """
    Run vector search and keyword search,
    combine the results, and remove duplicates.
    """

    # Step 1: Vector search
    vector_results = retrieve_chunks(
        question,
        top_k=vector_top_k
    )

    for result in vector_results:
        result["search_type"] = "vector"

    # Step 2: Keyword search
    keyword_results = keyword_search(
        question,
        top_k=keyword_top_k
    )

    # Step 3: Combine results
    combined_results = (
        vector_results
        + keyword_results
    )

    # Step 4: Remove duplicates
    unique_results = []
    seen_chunks = set()

    for result in combined_results:

        source = result.get("source", "")
        page = result.get("page", "")
        content = result.get("content", "")

        # Solr may return fields as lists
        if isinstance(source, list):
            source = source[0] if source else ""

        if isinstance(page, list):
            page = page[0] if page else ""

        if isinstance(content, list):
            content = content[0] if content else ""

        # Update result with normalized values
        result["source"] = source
        result["page"] = page
        result["content"] = content

        unique_key = (
            str(source),
            str(page),
            str(content)
        )

        if unique_key not in seen_chunks:
            seen_chunks.add(unique_key)
            unique_results.append(result)

    return unique_results[:hybrid_top_k]


print("✅ Hybrid search function created")

✅ Hybrid search function created


In [24]:
# ============================================================
# Cell 9 - Load Ollama LLM
# ============================================================

llm = ChatOllama(
    model=OLLAMA_MODEL_NAME,
    temperature=0.3)

print("✅ Ollama model loaded successfully")

✅ Ollama model loaded successfully


In [25]:
# # ============================================================
# # Cell 10 - RAG Question Answering Function
# # ============================================================

# def ask_rag(question, top_k=TOP_K):

#     # 1. Retrieve relevant chunks from Solr
#     retrieved_chunks = retrieve_chunks(question, top_k)

#     if not retrieved_chunks:
#         return {
#             "answer": "No relevant information was found in the PDF.",
#             "sources": []
#         }

#     # 2. Combine retrieved chunks into context
#     context_parts = []

#     for index, chunk in enumerate(retrieved_chunks, start=1):

#         source = chunk.get("source", "Unknown")
#         page = chunk.get("page", "Unknown")
#         content = chunk.get("content", "")

#         context_parts.append(
#             f"""Chunk {index}
# Source: {source}
# Page: {page}
# Content:
# {content}"""
#         )

#     context = "\n\n".join(context_parts)

#     # 3. Create the prompt
#     prompt = f"""
# You are a question-answering assistant.

# Answer the question using only the context provided below.

# Rules:
# - Do not use information outside the context.
# - If the answer is not present in the context, say:
#   "The answer is not available in the provided documents."
# - Give a clear and concise answer.
# - Do not mention chunk numbers in the answer.

# Context:
# {context}

# Question:
# {question}

# Answer:
# """

#     # 4. Send the prompt to Qwen through Ollama
#     response = llm.invoke(prompt)

#     # 5. Extract the answer
#     answer = response.content

#     # 6. Collect unique sources
#     sources = []

#     for chunk in retrieved_chunks:

#         source_info = {
#             "source": chunk.get("source", "Unknown"),
#             "page": chunk.get("page", "Unknown")
#         }

#         if source_info not in sources:
#             sources.append(source_info)

#     return {
#         "answer": answer,
#         "sources": sources
#     }


# print("✅ RAG question-answering function created")

In [ ]:
# # ============================================================
# # Cell 11 - Interactive RAG Question Loop
# # ============================================================

# print("=" * 70)
# print("PDF RAG Chat Started")
# print("Type 'exit' or 'esc' to stop.")
# print("=" * 70)

# while True:

#     question = input("\nAsk a question: ").strip()

#     # Stop the loop
#     if question.lower() in ["exit", "esc"]:
#         print("\nRAG chat stopped.Thank you for using the PDF RAG chat.")
#         break

#     # Prevent empty questions
#     if not question:
#         print("Please enter a question.")
#         continue

#     try:
#         print("\nSearching the PDF and generating the answer...\n")

#         result = ask_rag(question)

#         # Display answer
#         print("=" * 70)
#         print("ANSWER")
#         print("=" * 70)
#         print(result["answer"])

#         # Display sources
#         if result["sources"]:

#             print("\n" + "=" * 70)
#             print("SOURCES")
#             print("=" * 70)

#             for index, source in enumerate(result["sources"], start=1):
#                 print(
#                     f"{index}. Document: {source['source']} | "
#                     f"Page: {source['page']}"
#                 )

#     except Exception as error:
#         print("\nAn error occurred:", error)

PDF RAG Chat Started
Type 'exit' or 'esc' to stop.

Searching the PDF and generating the answer...

ANSWER
Drilling mud is a fluid used in drilling, completion, and workover processes that can be oil-based, water-based, or synthetic. It consists of powdered materials and is commonly referred to as mud. Its primary functions include matching desired flow thickness, lubrication properties, and specific gravity for various applications in the oil and gas industry.

SOURCES
1. Document: Oil and gas production handbook ed3x0_web.pdf | Page: 36
2. Document: Oil and gas production handbook ed3x0_web.pdf | Page: 35
Please enter a question.

RAG chat stopped.Thank you for using the PDF RAG chat.


In [26]:
# ============================================================
# Cell 12 - RunnableWithMessageHistory
# ============================================================

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

print("✅ Memory libraries imported")

✅ Memory libraries imported


In [27]:
# ============================================================
# Cell 13 - Create Chat History
# ============================================================

store = {}


def get_session_history(session_id: str):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]


print("✅ Conversation memory initialized")

✅ Conversation memory initialized


In [28]:
# ============================================================
# History-Aware Question Rewriter
# ============================================================

from langchain_core.messages import HumanMessage, AIMessage


def format_chat_history(history):
    """
    Convert LangChain chat messages into readable text.
    """

    formatted_history = []

    for message in history:

        if isinstance(message, HumanMessage):
            formatted_history.append(
                f"User: {message.content}"
            )

        elif isinstance(message, AIMessage):
            formatted_history.append(
                f"Assistant: {message.content}"
            )

    return "\n".join(formatted_history)


def rewrite_question(question, history):
    """
    Rewrite a follow-up question as a standalone question.
    """

    # No history means no rewriting is required
    if not history:
        return question

    history_text = format_chat_history(history)

    rewrite_prompt = f"""
You rewrite follow-up questions into standalone questions.

Use the conversation history only to understand references such as:
it, its, they, them, this, that, these, those, he and she.

Rules:
- Preserve the original meaning.
- Do not answer the question.
- Return only the rewritten standalone question.
- If the question is already standalone, return it unchanged.

Conversation history:
{history_text}

Current question:
{question}

Standalone question:
"""

    response = llm.invoke(rewrite_prompt)

    rewritten_question = response.content.strip()

    # Fallback if the model returns an empty response
    if not rewritten_question:
        return question

    return rewritten_question



# ============================================================
# Test Question Rewriting
# ============================================================

test_history = [
    HumanMessage(
        content="What is a wellhead?"
    ),
    AIMessage(
        content="A wellhead is equipment installed at the surface of a well."
    )
]

test_question = "What are its components?"

rewritten = rewrite_question(
    test_question,
    test_history
)

print("Original question:")
print(test_question)

print("\nRewritten question:")
print(rewritten)

Original question:
What are its components?

Rewritten question:
What are the components of a wellhead?


In [58]:
# ============================================================
# Cell 14 - Create Conversational Hybrid RAG Chain
# ============================================================

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser


# ============================================================
# Retrieve, Rerank and Prepare Context
# ============================================================

# ============================================================
# History-Aware Context Retrieval
# ============================================================

def get_context(inputs):

    # Original user question
    question = inputs["question"]

    # Previous conversation messages
    history = inputs.get("history", [])

    # Step 1: Rewrite follow-up question
    rewritten_question = rewrite_question(
        question,
        history
    )

    print("\nOriginal question:")
    print(question)

    print("\nRetrieval question:")
    print(rewritten_question)

    # Step 2: Hybrid retrieval using rewritten question
    retrieved_chunks = hybrid_search(
        rewritten_question
    )

    # Step 3: Rerank using rewritten question
    reranked_chunks = rerank_chunks(
        rewritten_question,
        retrieved_chunks,
        top_k=RERANK_TOP_K
    )

    # Step 4: Build context
    context = "\n\n".join(
        chunk.get("content", "")
        for chunk in reranked_chunks
    )

    # Step 5: Prepare source details
    sources = []

    for chunk in reranked_chunks:
        sources.append({
            "source": chunk.get(
                "source",
                "Unknown source"
            ),
            "page": chunk.get(
                "page",
                "Unknown page"
            ),
            "search_type": chunk.get(
                "search_type",
                "Unknown"
            ),
            "reranker_score": chunk.get(
                "reranker_score",
                0
            )
        })

    return {
        # Keep the original question for final answer generation
        "question": question,

        # Retrieved using the rewritten question
        "context": context,

        # Keep conversation history in the final prompt
        "history": history,

        # Optional debugging information
        "rewritten_question": rewritten_question,

        "sources": sources
    }
# ============================================================
# Prompt Template
# ============================================================

#The required context is not provided, so I cannot answer this question.
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a question-answering assistant.

Answer the question using only the context provided below, except for the special rule listed below.

Rules:
- If the user asks "Who is Abid?" or asks an equivalent question about Abid, reply exactly:
  "He is the owner."
- If the user asks "Who is ravi?" or asks an equivalent question about ravi, reply exactly:
  "He is a a good boy."
- For all other questions, use only the provided context.
- If the provided context is empty or does not contain relevant information, reply exactly:
  "The required context is not provided, so I cannot answer this question."
- Give a clear and concise answer.
- Do not mention chunk numbers.

Context:
{context}
"""
        ),

        MessagesPlaceholder(
            variable_name="history"
        ),

        (
            "human",
            """
Question:
{question}

Answer:
"""
        ),
    ]
)
# ============================================================
# Build Hybrid RAG Chain
# ============================================================

rag_chain = (
    RunnableLambda(get_context)
    | prompt
    | llm
    | StrOutputParser()
)


print(
    "✅ Conversational hybrid RAG chain with reranking created"
)

✅ Conversational hybrid RAG chain with reranking created


In [53]:
# ============================================================
# Cell 15 - Add RunnableWithMessageHistory
# ============================================================

conversational_rag = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

print("✅ Conversational memory enabled")

✅ Conversational memory enabled


C:\Users\AbidDileepJan\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [57]:
# ============================================================
# Test History-Aware Conversational RAG
# ============================================================

SESSION_ID = "history_test_1"

print("History-aware conversational RAG started.")
print("Type 'exit' or 'esc' to stop.\n")

while True:

    question = input("Ask a question: ").strip()

    if question.lower() in ["exit", "esc"]:
        print("Chat stopped.")
        break

    if not question:
        print("Please enter a question.\n")
        continue

    try:

        answer = conversational_rag.invoke(
            {
                "question": question
            },
            config={
                "configurable": {
                    "session_id": SESSION_ID
                }
            }
        )

        print("\nAnswer:")
        print(answer)
        print()

    except Exception as error:
        print("\nError:", error)
        print()

History-aware conversational RAG started.
Type 'exit' or 'esc' to stop.


Original question:
who is abid

Retrieval question:
Who is Abid?

Answer:
He is the owner.

Chat stopped.


In [3]:
# ============================================================
# Cell 16 - Create Response Cache
# ============================================================

response_cache = {}

print("✅ Response cache created")

✅ Response cache created


In [45]:
# ============================================================
# Cell 17 - Conversational RAG with Cache and Sources
# ============================================================

SESSION_ID = "user_1"

print("Conversational RAG is ready.")
print("Type 'exit' or 'esc' to stop.\n")


def normalize_question(question):
    """
    Normalize the question so small differences in capitalization
    and extra spaces use the same cache entry.
    """
    return " ".join(question.lower().strip().split())


while True:

    question = input("You: ").strip()

    if question.lower() in ["exit", "esc"]:
        print("Chat ended.")
        break

    if not question:
        continue

    try:

        cache_key = normalize_question(question)

        # ----------------------------------------------------
        # Check cache
        # ----------------------------------------------------
        if cache_key in response_cache:

            cached_result = response_cache[cache_key]

            answer = cached_result["answer"]
            sources = cached_result["sources"]

            print("\n⚡ Answer loaded from cache")

        else:

            print("\nSearching and generating answer...\n")

            # Generate answer using conversational memory
            answer = conversational_rag.invoke(
                {
                    "question": question
                },
                config={
                    "configurable": {
                        "session_id": SESSION_ID
                    }
                }
            )

            # Retrieve sources
            retrieved_chunks = retrieve_chunks(
                question,
                top_k=RETRIEVAL_TOP_K
            )

            sources = []

            for chunk in retrieved_chunks:

                source = chunk.get("source", "Unknown")
                page = chunk.get("page", "Unknown")

                if isinstance(source, list):
                    source = source[0] if source else "Unknown"

                if isinstance(page, list):
                    page = page[0] if page else "Unknown"

                sources.append(
                    {
                        "source": source,
                        "page": page
                    }
                )

            # Save answer and sources in cache
            response_cache[cache_key] = {
                "answer": answer,
                "sources": sources
            }

            print("✅ Answer saved to cache")

        # ----------------------------------------------------
        # Display answer
        # ----------------------------------------------------
        print("\n" + "=" * 70)
        print("ANSWER")
        print("=" * 70)
        print(answer)

        print("\n" + "=" * 70)
        print("SOURCES")
        print("=" * 70)

        for index, item in enumerate(sources, start=1):
            print(
                f"{index}. Document: {item['source']} | "
                f"Page: {item['page']}"
            )

        print()

    except Exception as error:
        print(f"\nError: {error}\n")

Conversational RAG is ready.
Type 'exit' or 'esc' to stop.


Searching and generating answer...


Original question:
who is adarsh

Retrieval question:
adarsh是谁
✅ Answer saved to cache

ANSWER
I don't have enough information to determine who Adarsh is based on the provided context.

SOURCES
1. Document: Oil and gas production handbook ed3x0_web.pdf | Page: 100
2. Document: Oil and gas production handbook ed3x0_web.pdf | Page: 104
3. Document: Oil and gas production handbook ed3x0_web.pdf | Page: 79
4. Document: Oil and gas production handbook ed3x0_web.pdf | Page: 90
5. Document: Oil and gas production handbook ed3x0_web.pdf | Page: 71
6. Document: Oil and gas production handbook ed3x0_web.pdf | Page: 104

Chat ended.


In [4]:
# ============================================================
# View Response Cache
# ============================================================

print("=" * 80)
print("RESPONSE CACHE")
print("=" * 80)

if not response_cache:
    print("Cache is empty.")

else:

    print(f"Total Cached Questions: {len(response_cache)}\n")

    for index, (question, data) in enumerate(response_cache.items(), start=1):

        print(f"{index}. Question")
        print("-" * 80)
        print(question)

        print("\nAnswer")
        print("-" * 80)
        print(data["answer"])

        print("\nSources")
        print("-" * 80)

        for source in data["sources"]:
            print(
                f"Document: {source['source']} | "
                f"Page: {source['page']}"
            )

        print("\n" + "=" * 80 + "\n")

RESPONSE CACHE
Cache is empty.


In [5]:
# ============================================================
# Clear Response Cache
# ============================================================

response_cache.clear()

print("✅ Response cache has been cleared.")
print(f"Current Cache Size: {len(response_cache)}")

✅ Response cache has been cleared.
Current Cache Size: 0
